<a href="https://colab.research.google.com/github/chhammet/ST554_HW6/blob/main/cole_hammett_hw_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 6 — ST 554: Big Data with Python
**Author:** Cole Hammett (`chhammet@ncsu.edu`)

This notebook covers two parts:
- **Part I** – SQL queries against the Lahman baseball SQLite database
- **Part II** – Demonstration of the `SLR_slope_simulator` class defined in `SLR_slope_simulator.py`

---
## Part I — More Practice Querying a Database

The [Lahman Baseball Database](http://www.seanlahman.com/baseball-archive/statistics/) contains historical Major League Baseball statistics.

In [9]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from numpy.random import default_rng
from sklearn import linear_model

print('Libraries imported successfully.')

Libraries imported successfully.


### Loading data





1)  Connected to the database by uploading to Colab

In [10]:
con = sqlite3.connect("lahman_1871-2022.sqlite")

2) Looked at all of the tables in the
database (as a data frame) using `read_sql()` from pandas.

In [11]:

pd.read_sql("SELECT * FROM sqlite_schema WHERE type = 'table';", con)

,type,name,tbl_name,rootpage,sql
0,table,AllstarFull,AllstarFull,2,"CREATE TABLE AllstarFull (\nplayerID TEXT,\nye..."
1,table,Appearances,Appearances,3,"CREATE TABLE Appearances (\nyearID INTEGER,\nt..."
2,table,AwardsManagers,AwardsManagers,4,"CREATE TABLE AwardsManagers (\nplayerID TEXT,\..."
3,table,AwardsPlayers,AwardsPlayers,5,"CREATE TABLE AwardsPlayers (\nplayerID TEXT,\n..."
4,table,AwardsShareManagers,AwardsShareManagers,6,CREATE TABLE AwardsShareManagers (\nawardID TE...
5,table,AwardsSharePlayers,AwardsSharePlayers,7,CREATE TABLE AwardsSharePlayers (\nawardID TEX...
6,table,Batting,Batting,8,"CREATE TABLE Batting (\nplayerID TEXT,\nyearID..."
7,table,BattingPost,BattingPost,9,"CREATE TABLE BattingPost (\nyearID INTEGER,\nr..."
8,table,CollegePlaying,CollegePlaying,10,"CREATE TABLE CollegePlaying (\nplayerID TEXT,\..."
9,table,Fielding,Fielding,11,"CREATE TABLE Fielding (\nplayerID TEXT,\nyearI..."


###  Constructing a table of hall of fame (HoF) pitchers
by joining `HallOfFame` table with the `Pitching` table

1) Pulled all HoF inductees (inducted = 'Y') into Python




In [12]:
hof_df = pd.read_sql(
    "SELECT DISTINCT playerID FROM HallOfFame WHERE inducted = 'Y';",
    con
)

2) Pulled the full Pitching table

In [13]:
pitching_df = pd.read_sql("SELECT * FROM Pitching;", con)

3) Filtered to only pitchers who are in the HoF

In [14]:
hof_pitchers_raw = pitching_df[pitching_df['playerID'].isin(hof_df['playerID'])]

4) Summed the requested pitching columns per player: `GS`, `G`, `W`, `L`, `IPouts`, `CG`, `SHO`, `SV`

In [15]:
pitching_stats = (
    hof_pitchers_raw
    .groupby('playerID')[['GS', 'G', 'W', 'L', 'IPouts', 'CG', 'SHO', 'SV']]
    .sum()
    .reset_index()
)

print(f'Hall of Fame pitchers found: {len(pitching_stats)}')
pitching_stats.head(10)

Hall of Fame pitchers found: 108


,playerID,GS,G,W,L,IPouts,CG,SHO,SV
0,alexape01,599,696,373,208,15570,437,90,32
1,ansonca01,0,3,0,1,12,0,0,1
2,becklja01,1,1,0,1,12,0,0,0
3,bendech01,334,459,212,127,9051,255,40,34
4,blylebe01,685,692,287,250,14910,242,60,0
5,boggswa01,0,2,0,0,7,0,0,0
6,bresnro01,6,9,4,1,151,3,1,0
7,broutda01,2,4,0,2,69,2,0,0
8,brownmo01,332,481,239,130,9517,271,55,49
9,bunniji01,519,591,224,184,11281,151,40,16


### Creating a table of HoF pitcher *batting* statistics

1) Pulled the full Batting table

In [16]:
batting_df = pd.read_sql("SELECT * FROM Batting;", con)


2) Kept only rows for HoF pitchers

In [17]:
hof_batting_raw = batting_df[batting_df['playerID'].isin(pitching_stats['playerID'])]


3) Summed the batting columns per player: `AB`, `R`, `H`, `HR`, `RBI`, `BB`, `SO`

In [18]:
batting_stats = (
    hof_batting_raw
    .groupby('playerID')[['AB', 'R', 'H', 'HR', 'RBI', 'BB', 'SO']]
    .sum()
    .reset_index()
)

print(f'Hall of Fame pitchers with batting records: {len(batting_stats)}')
batting_stats.head(10)

Hall of Fame pitchers with batting records: 108


,playerID,AB,R,H,HR,RBI,BB,SO
0,alexape01,1810,154,378,11,163.0,77,276.0
1,ansonca01,10281,1999,3435,97,2075.0,984,330.0
2,becklja01,9551,1603,2938,87,1581.0,616,526.0
3,bendech01,1147,102,243,6,116.0,75,143.0
4,blylebe01,451,19,59,0,25.0,5,193.0
5,boggswa01,9180,1513,3010,118,1014.0,1412,745.0
6,bresnro01,4481,682,1252,26,530.0,714,403.0
7,broutda01,6726,1529,2303,107,1301.0,840,238.0
8,brownmo01,1143,93,235,2,74.0,44,199.0
9,bunniji01,1275,82,213,7,75.0,34,362.0


### Joining pitching and batting tables
to generate a single combined table of pitching and batting stats for each HoF pitcher

1) Merged pitching and batting summaries on playerID using an inner join

In [19]:
combined_df = pd.merge(
    pitching_stats,
    batting_stats,
    on='playerID',
    how='inner'
)

print(f'Combined table shape: {combined_df.shape}')
combined_df.head(10)

Combined table shape: (108, 16)


,playerID,GS,G,W,L,IPouts,CG,SHO,SV,AB,R,H,HR,RBI,BB,SO
0,alexape01,599,696,373,208,15570,437,90,32,1810,154,378,11,163.0,77,276.0
1,ansonca01,0,3,0,1,12,0,0,1,10281,1999,3435,97,2075.0,984,330.0
2,becklja01,1,1,0,1,12,0,0,0,9551,1603,2938,87,1581.0,616,526.0
3,bendech01,334,459,212,127,9051,255,40,34,1147,102,243,6,116.0,75,143.0
4,blylebe01,685,692,287,250,14910,242,60,0,451,19,59,0,25.0,5,193.0
5,boggswa01,0,2,0,0,7,0,0,0,9180,1513,3010,118,1014.0,1412,745.0
6,bresnro01,6,9,4,1,151,3,1,0,4481,682,1252,26,530.0,714,403.0
7,broutda01,2,4,0,2,69,2,0,0,6726,1529,2303,107,1301.0,840,238.0
8,brownmo01,332,481,239,130,9517,271,55,49,1143,93,235,2,74.0,44,199.0
9,bunniji01,519,591,224,184,11281,151,40,16,1275,82,213,7,75.0,34,362.0
